# Tutorial 16: Hallucination Detection via Geodesic Distance

**Conceptual deep-dive:** `gmsh_testing_ext/chapters/ch16.tex` — Hallucination Detection via Geodesic Distance
**Monograph notebook:** `gmsh_testing_ext/notebooks/ch16_hallucination_detection.ipynb`
**Companion guide section:** `knowlytix/harness/testing/USER_GUIDE.md` §19
**Prereqs:** NB 02 (Primitives), NB 09 (Verifier Routing), NB 20 (Calibration)

**What you'll do.** Build a `HallucinationOracle` over the bank model-risk store and test it on three handcrafted claims: a true triple `(MDL-001, has_test_auc, 0.8412)`, a single-edit hallucination `(MDL-001, has_test_auc, 0.9999)`, and a wrong-model hallucination `(MDL-007, has_test_auc, 0.8412)`. Read off the geodesic confidence for each, see how the calibrated τ_geo threshold separates them, and watch the `HALLUCINATION_DETECTED` failure code attach to the synthetic hallucinations.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from _shared import (
    CH04_STORE_PATH,
    LAB_STORES_PATH,
    REPO_ROOT,
    load_ch04_store,
    seed_all,
)

from knowlytix.harness.testing import HallucinationOracle

seed_all()
import os

os.chdir(REPO_ROOT)
store = load_ch04_store()
print(f"store: {store.stats()['entities']} entities, {store.stats()['triples']} triples")


## 2. Concept

`HallucinationOracle.assess_claim(head, relation, tail)` returns a Chapter-9 `SubVerdict`. Internally:

1. Call `store.score_triple(head, relation, tail)` to get the geodesic distance on the cap manifold.
2. Compare against `tau_geodesic` (passed at construction or loaded from `store/calibration.json`).
3. If `geodesic < tau_geo`, the triple is plausible → `passed=True`; otherwise `HALLUCINATION_DETECTED`.

The trained calibration artifact provides `tau_geo` per corpus. Without it, the oracle falls back to a conservative default (the chapter explains the choice). On the bank ch04 corpus, NB 20 produced `τ_geodesic = 0.876`.

## 3. Train Calibration and Build the Oracle

In [ ]:
import contextlib
import io

buf = io.StringIO()
# We need the GMSJudge from a benchmark to run calibrate(), which produces τ_geo.
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(CH04_STORE_PATH / "documents" / "combined.md"),
    stores_dir=str(LAB_STORES_PATH),
    store_name="ch04_ground_truth",
    factor_group="quick_screen", n_runs=1, cap_enabled=True,
    enable_typed_verdicts=False, enable_tracing=False,
)
bench = DOEGMSBenchmark(config)
with contextlib.redirect_stdout(buf):
    bench.ingest()
    bench.judge.calibrate(seed=42)
tau_geo = bench.judge._thresholds.get("geodesic")
print(f"calibrated tau_geo: {tau_geo:.4f}")
oracle = HallucinationOracle(store=store, tau_geodesic=tau_geo)
print(f"oracle built with tau_geodesic={oracle.tau_geodesic:.4f}")


## 4. Three Handcrafted Claims

In [ ]:
claims = [
    ("mdl-001", "has_test_auc", "0.8412",
     "TRUE — MDL-001 test AUC matches ground truth"),
    ("mdl-001", "has_test_auc", "0.9999",
     "HALLUCINATION — wrong AUC value for the right model"),
    ("mdl-007", "has_test_auc", "0.8412",
     "HALLUCINATION — wrong model attributed with MDL-001's AUC"),
]

print(f"{'head':10s}  {'relation':18s}  {'tail':12s}  {'verdict':9s}  {'geo':>7s}  note")
print("-" * 110)
for h, r, t, note in claims:
    sv = oracle.assess_claim(h, r, t)
    geo = sv.raw_score if sv.raw_score is not None else float("nan")
    verdict = "PASS" if sv.passed else "FAIL"
    print(f"{h:10s}  {r:18s}  {t:12s}  {verdict:9s}  {geo:>7.4f}  {note}")

## 5. Visualization — Geodesic Distribution vs τ_geo

In [ ]:
import matplotlib.pyplot as plt

# Sample geodesic scores for a sweep of true triples vs random corruptions
true_scores = []
for h, r, t in list(store.doc_graph.triples)[:200]:
    s = store.score_triple(h, r, t)
    if s is not None:
        true_scores.append(s)

import random

rng = random.Random(42)
entities = list(store.adapter.entity_to_idx.keys())
fake_scores = []
for h, r, _ in list(store.doc_graph.triples)[:200]:
    fake_t = rng.choice(entities)
    s = store.score_triple(h, r, fake_t)
    if s is not None:
        fake_scores.append(s)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(true_scores, bins=30, alpha=0.6, color="#3aa56b", label=f"true triples (n={len(true_scores)})")
ax.hist(fake_scores, bins=30, alpha=0.6, color="#c14a4a", label=f"hallucinated triples (n={len(fake_scores)})")
ax.axvline(tau_geo, color="black", linestyle="--", linewidth=2, label=f"τ_geo = {tau_geo:.3f}")
ax.set_xlabel("geodesic distance")
ax.set_ylabel("count")
ax.set_title("Geodesic-distance distribution: true vs hallucinated triples (bank corpus)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"true triples below τ_geo:        {sum(s < tau_geo for s in true_scores):>3d} / {len(true_scores)}")
print(f"hallucinated triples below τ_geo: {sum(s < tau_geo for s in fake_scores):>3d} / {len(fake_scores)}")

## 6. Take-Aways

1. **τ_geo is a learned cut.** Don't hand-tune it; let `bench.judge.calibrate()` derive it from the corpus's positive/negative pairs.
2. **`HALLUCINATION_DETECTED` is `CRITICAL` at action_taking.** NB 22 spells out the tier escalation; a single hallucinated claim under an autonomous credit decision blocks the gate.
3. **The histogram is the calibration sanity check.** True and hallucinated distributions should separate cleanly around τ_geo. Heavy overlap means the embedding hasn't learned enough discrimination — go retrain.
4. **The oracle works on triples, not free text.** Pair with `ClaimCompiler` (NB 08) to extract triples from an LLM answer first; the oracle then judges each one individually.
5. **`evaluate(scenario, trace)` is the release-gate hook.** When pairing with `TieredReleaseGate` (NB 19), the oracle returns an `OracleVerdict` with the failure list pre-populated — exactly the shape the audit reporter consumes.

## 7. Next Steps

- **NB 17** — `17_multi_turn.ipynb` — apply hallucination detection across a multi-turn dialog (Ch17)
- **NB 22** — `22_failure_codes.ipynb` — `HALLUCINATION_DETECTED` in the failure-code taxonomy (App C)

For theory: `gmsh_testing_ext/chapters/ch16.tex`.